# 01 · Exploración inicial de los CSV
Solo exploración. Lee `data/raw/`, muestra el contenido y guarda una copia **sin limpiar** en `data/interim/`.

## Origen de los datos
- **Fuente:** [Accidentes de tráfico de la ciudad de Madrid](https://datos.madrid.es/dataset/300228-0-accidentes-trafico-detalle) (Ayuntamiento de Madrid / Policía Municipal).
- **Ficheros:** `accidentes_trafico_2019.csv` … `accidentes_trafico_2025.csv`.
- **Fecha de descarga:** 9 de noviembre de 2026.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT / "src"))

from load_data import cargar_raw

INTERIM = ROOT / "data" / "interim"
pd.set_option("display.max_columns", None)

## Carga

In [ ]:
datos = cargar_raw()
datos.shape

In [ ]:
datos.head()

## Filas (personas) y expedientes (accidentes) por año

In [ ]:
resumen = datos.groupby("año").agg(
    filas=("num_expediente", "size"),
    expedientes=("num_expediente", "nunique"),
)
resumen.loc["total"] = [len(datos), datos["num_expediente"].nunique()]
resumen

## Nulos
Aparecen como cadena vacía (`NaN` al leer) y como el texto `NULL`.

In [ ]:
nulos = datos.isna().sum().add((datos == "NULL").sum())
nulos[nulos > 0].sort_values(ascending=False)

## Guardado en `data/interim/`
Copia tal cual (todo texto, sin normalizar).

In [ ]:
salida = INTERIM / "accidentes_raw_concat.parquet"
datos.to_parquet(salida, index=False)
print(salida, f"{salida.stat().st_size / 1e6:.1f} MB")

## Normalización de nulos y tipos
La lógica está en `src/normalize.py` (`normalizar`). Aquí solo se llama y se comprueba el resultado. **Todavía no se guarda nada:** el Parquet limpio vendrá después de consolidar los expedientes.

In [ ]:
from normalize import normalizar

limpio = normalizar(datos)
limpio.dtypes

### Comprobación: los nulos tienen que coincidir con los de antes
Antes había tres formas de decir «sin dato» (vacío, `NULL` y `#¡VALOR!`); ahora solo `NaN`.

In [ ]:
antes = datos.isna().sum().add((datos == "NULL").sum()).add((datos == "#¡VALOR!").sum())
despues = limpio.isna().sum()
comparacion = pd.DataFrame({"antes": antes, "despues": despues})
comparacion["igual"] = comparacion["antes"] == comparacion["despues"]
comparacion[(comparacion["antes"] > 0) | ~comparacion["igual"]]

In [ ]:
assert comparacion["igual"].all(), "Los nulos no coinciden"
assert len(limpio) == len(datos)
print("OK: mismos nulos y mismas filas")

### Vistazo a los tipos nuevos

In [ ]:
limpio[["fecha", "hora", "coordenada_x_utm", "coordenada_y_utm", "cod_distrito", "cod_lesividad"]].describe(include="all")

## Un accidente por fila y cruce con el contorno de Madrid
La lógica está en `src/geografia.py`. El contorno (`data/external/`) viene del Ayuntamiento de Madrid (CC BY 4.0) y está en EPSG:25830. **No se descarta ningún accidente ni se guarda nada todavía:** solo se miden los problemas.

In [ ]:
from geografia import a_expedientes, cruce_espacial

expedientes = a_expedientes(limpio)
print(len(datos), "filas (personas) ->", len(expedientes), "expedientes")
assert expedientes["num_expediente"].is_unique

In [ ]:
cruce = cruce_espacial(expedientes)
estado = cruce.apply(
    lambda f: "sin coordenadas" if f["sin_coord"] else ("dentro" if f["dentro_municipio"] else "fuera"), axis=1
)
estado.value_counts()

### Por año

In [ ]:
pd.crosstab(cruce["año"], estado, margins=True)

### Los que caen fuera: ¿lejos o pegados al límite?
Si fueran coordenadas mal escaladas estarían a kilómetros; si son del borde, a pocos metros.

In [ ]:
fuera = cruce[estado == "fuera"]
fuera["distancia_limite_m"].describe().round(1)

In [ ]:
fuera[["num_expediente", "localizacion", "coordenada_x_utm", "coordenada_y_utm", "distancia_limite_m"]].sort_values("distancia_limite_m", ascending=False).head(10)

### Distrito del CSV frente a distrito según la posición
No tienen por qué coincidir al 100 %: un accidente en una calle fronteriza puede asignarse al distrito vecino.

In [ ]:
comp = cruce.dropna(subset=["cod_distrito", "distrito_geo"]).copy()
comp["coincide"] = comp["cod_distrito"] == comp["distrito_geo"]
print(f"coinciden: {comp['coincide'].mean():.1%} ({(~comp['coincide']).sum()} difieren de {len(comp)})")
comp.groupby("año")["coincide"].mean().round(3)

## Cierre de Etapa 1: `accidents_clean.parquet`
Decisiones aplicadas se **conservan** los accidentes de borde (fuera del límite pero a pocos metros), se **excluyen** los que no tienen coordenadas y para E5 se usa el **distrito geográfico**.

In [ ]:
from geografia import accidentes_limpios, informe_perdidas

informe = informe_perdidas(cruce)
informe

In [ ]:
limpios = accidentes_limpios(cruce)
salida = INTERIM / "accidents_clean.parquet"
limpios.to_parquet(salida, index=False)
informe.to_csv(INTERIM / "informe_perdidas_m1.csv")
print(salida, f"{salida.stat().st_size / 1e6:.1f} MB", len(limpios), "expedientes")

### Comprobación: se relee el Parquet y se contrasta con el informe

In [ ]:
leido = pd.read_parquet(salida)
assert len(leido) == informe.loc["total", "en_accidents_clean"]
assert leido["num_expediente"].is_unique
assert leido[["coordenada_x_utm", "coordenada_y_utm"]].notna().all().all()
assert leido["distrito_geo"].notna().sum() == leido["dentro_municipio"].sum()
print("OK")
leido.dtypes

# Etapa 2 · Mallas
La lógica está en `src/mallas.py`. Las celdas salen **solo del contorno** del municipio (sin mirar accidentes) y entra toda celda que solape con Madrid, aunque lo cubra en parte. Cada resolución se guarda aparte en GeoParquet.

In [ ]:
from mallas import crear_malla, malla_distritos, guardar_mallas

malla_1km = crear_malla(1000)
malla_2km = crear_malla(2000)
malla_dist = malla_distritos()
pd.DataFrame({"celdas": [len(malla_1km), len(malla_2km), len(malla_dist)]}, index=["1 km", "2 km", "distritos"])

### ¿Cuánto cae dentro de Madrid? (celdas de borde)

In [ ]:
resumen = pd.DataFrame({
    "1 km": malla_1km["frac_dentro"].describe(),
    "2 km": malla_2km["frac_dentro"].describe(),
}).round(2)
resumen.loc["celdas < 50 % dentro"] = [(malla_1km["frac_dentro"] < .5).sum(), (malla_2km["frac_dentro"] < .5).sum()]
resumen

In [ ]:
import matplotlib.pyplot as plt
import geopandas as gpd
from geografia import TERMINO

fig, ax = plt.subplots(figsize=(7, 7))
malla_1km.plot(ax=ax, column="frac_dentro", cmap="Blues", edgecolor="grey", linewidth=0.3, legend=True)
gpd.read_file(TERMINO).boundary.plot(ax=ax, color="red", linewidth=1)
ax.set_title("Malla de 1 km (color = fracción dentro de Madrid)")
ax.set_axis_off()

### Comprobación con los accidentes
El cruce geométrico estricto deja sin celda a los accidentes que caen **justo sobre una línea** de la malla (las coordenadas son metros enteros, así que X o Y terminadas en `000` coinciden con el borde). Hay que fijar una regla de desempate en la asignación.

In [ ]:
limpios = pd.read_parquet(INTERIM / "accidents_clean.parquet")
sobre_linea = (limpios["coordenada_x_utm"] % 1000 == 0) | (limpios["coordenada_y_utm"] % 1000 == 0)
print("accidentes exactamente sobre una línea de la malla de 1 km:", int(sobre_linea.sum()))
print("accidentes de borde (fuera del límite):", int((~limpios["dentro_municipio"]).sum()))

### Guardado
Una malla por resolución en `data/interim/` (GeoParquet, con geometría). Se regeneran con este notebook.

In [ ]:
rutas = guardar_mallas()
rutas

### Asignación de cada accidente a su celda
Regla de desempate: **intervalo semiabierto** (`x_min <= x < x_min + lado`, igual en Y). Un accidente justo sobre una línea va a la celda de su este/norte, así que nunca queda sin celda ni en dos. Los que caen fuera de todas las celdas (borde del municipio) van a la **más cercana**.

In [ ]:
from mallas import asignar_celda_cuadrada, asignar_distrito

a1 = asignar_celda_cuadrada(limpios, malla_1km)
a2 = asignar_celda_cuadrada(limpios, malla_2km)
ad = asignar_distrito(limpios, malla_dist)

pd.DataFrame(
    {"asignados": [len(a1), len(a2), len(ad)], "por cercanía": [a1["por_cercania"].sum(), a2["por_cercania"].sum(), ad["por_cercania"].sum()],
     "celdas distintas": [a1["cell_id"].nunique(), a2["cell_id"].nunique(), ad["cell_id"].nunique()]},
    index=["1 km", "2 km", "distritos"],
)

In [ ]:
asignacion = pd.DataFrame({
    "num_expediente": limpios["num_expediente"],
    "fecha": limpios["fecha"],
    "mes": limpios["fecha"].dt.to_period("M").dt.to_timestamp(),
    "cell_1km": a1["cell_id"],
    "cell_2km": a2["cell_id"],
    "cell_distrito": ad["cell_id"],
    "por_cercania": a1["por_cercania"] | a2["por_cercania"] | ad["por_cercania"],
})

assert len(asignacion) == len(limpios) and asignacion["num_expediente"].is_unique
assert asignacion[["cell_1km", "cell_2km", "cell_distrito"]].notna().all().all()
assert asignacion["cell_1km"].isin(malla_1km["cell_id"]).all() and asignacion["cell_2km"].isin(malla_2km["cell_id"]).all()
print("OK: todos los accidentes tienen celda en las tres resoluciones; total", len(asignacion))

### Guardado
`data/interim/accidents_celdas.parquet`: una fila por accidente con su celda en cada resolución. Es la entrada del panel celda-mes.

In [ ]:
asignacion.to_parquet(INTERIM / "accidents_celdas.parquet", index=False)
asignacion.head()